In [ ]:
import marimo as mo

# Task 2 — Generate Product Descriptions

Using the rubric defined in Task 1, we generate a persuasive 50–90 word product
description for every product in the dataset using **Meta-Llama-3.1-8B-Instruct** on Nebius.

## Setup

In [ ]:
from _bootstrap import bootstrap_notebook
bootstrap_notebook()

import time
import litellm
import mlflow
import pandas as pd
from tqdm import tqdm

from src.artifacts import load_excel_artifact
from src.config import (
    DEFAULT_DEV_PROVIDER,
    PROMPT_VERSION_V1,
    get_force_rerun,
    get_generation_config,
    prompt_path,
)
from src.paths import ASSIGNMENT_XLSX_PATH, PRODUCTS_CSV_PATH
from src.runtime import load_project_env, read_text, setup_mlflow
from src.utils import extract_cost, format_product

load_project_env()
mlflow_db_path = setup_mlflow("product_descriptions")

In [ ]:
print(f"MLflow tracking DB: {mlflow_db_path}")

MLflow tracking DB: /Users/zakhar/Documents/code/learning-ai-nebius-ai-engineering-2026/w2-ai-product/outputs/experiments.db


## Model & Provider Configuration

- **Provider**: Nebius (for final deliverable runs)
- **Model**: `meta-llama/Meta-Llama-3.1-8B-Instruct`
- Change `PROVIDER` to `"nvidia_nim"` for free dev/testing (40 RPM limit)

In [ ]:
FORCE_RERUN = get_force_rerun()
generation_config = get_generation_config()
MODEL = generation_config.model
MODEL_ID = generation_config.model_id
PROVIDER = generation_config.provider
print(f"Using model: {MODEL}")
print(f"Set GENERATION_PROVIDER={DEFAULT_DEV_PROVIDER} for free dev/testing (40 RPM limit).")
if FORCE_RERUN:
    print("FORCE_RERUN=1 — ignoring existing Task 2 artifact.")

Using model: nebius/meta-llama/Meta-Llama-3.1-8B-Instruct
Set GENERATION_PROVIDER=nvidia_nim for free dev/testing (40 RPM limit).


## System Prompt

Designed to satisfy the rubric:
- Explicit word-count constraint (50–90)
- Grounding instruction: use ONLY provided data
- Tone instruction: friendly, credible sales voice
- Output format: description only, no labels

In [ ]:
SYSTEM_PROMPT = read_text(prompt_path(PROMPT_VERSION_V1))
print(SYSTEM_PROMPT)

You are a professional e-commerce copywriter. Your task is to write a persuasive product description based solely on the information provided.

## Rules
1. Length: write exactly 50–90 words. Count carefully before submitting.
2. Tone: friendly, credible sales voice. Highlight benefits, not just specs.
3. Grounding: use ONLY information from the product data provided. Do not invent features, add claims, or embellish beyond what is explicitly stated.
4. Fluency: write in natural, flowing English sentences. Avoid bullet points, lists, or fragmented phrases.
5. Grammar: use correct spelling and punctuation throughout.

## Output format
Return only the product description — no headings, no labels, no commentary.



## Generation Function

In [ ]:
def generate_description(product: dict) -> dict:
    """
    Call the LLM to generate a product description.
    Returns dict with generated_description, latency_ms, input_tokens, output_tokens, cost_usd.
    """
    start = time.perf_counter_ns()
    response = litellm.completion(
        model=MODEL,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user",   "content": format_product(product)},
        ],
        temperature=0.3,
        max_tokens=200,
    )
    latency_ms = (time.perf_counter_ns() - start) / 1e6

    input_tokens  = response.usage.prompt_tokens
    output_tokens = response.usage.completion_tokens
    cost_usd      = extract_cost(response, input_tokens, output_tokens)

    return {
        "generated_description": response.choices[0].message.content.strip(),
        "latency_ms":   round(latency_ms, 1),
        "input_tokens": input_tokens,
        "output_tokens": output_tokens,
        "cost_usd":     round(cost_usd, 8),
    }

## Run Generation on All Products

In [ ]:
required_columns = [
    "product_name",
    "generated_description",
    "latency_ms",
    "input_tokens",
    "output_tokens",
    "cost_usd",
    "fluency",
    "grammar",
    "tone",
    "length",
    "grounding",
    "latency",
    "cost",
    "final_score",
]
existing_df = None
if not FORCE_RERUN:
    existing_df = load_excel_artifact(
        ASSIGNMENT_XLSX_PATH,
        required_columns=required_columns,
    )

if existing_df is not None:
    print(f"Source: loaded existing artifact {ASSIGNMENT_XLSX_PATH}")
    df = existing_df.copy()
else:
    print(f"Source: artifact missing or invalid, running live generation -> {ASSIGNMENT_XLSX_PATH}")
    products = pd.read_csv(PRODUCTS_CSV_PATH)
    results = []
    with mlflow.start_run(run_name="generation_v1_llama8b"):
        mlflow.log_params({
            "model": "meta-llama/Meta-Llama-3.1-8B-Instruct",
            "provider": "nebius",
            "prompt_version": "v1",
            "temperature": 0.3,
            "max_tokens": 200,
        })

        for _, row in tqdm(products.iterrows(), total=len(products), desc="Generating"):
            result = generate_description(row.to_dict())
            results.append(result)

        results_df = pd.DataFrame(results)
        mlflow.log_metrics({
            "mean_latency_ms": results_df["latency_ms"].mean(),
            "total_cost_usd": results_df["cost_usd"].sum(),
            "mean_input_tokens": results_df["input_tokens"].mean(),
            "mean_output_tokens": results_df["output_tokens"].mean(),
        })

    df = pd.concat([products.reset_index(drop=True), results_df.reset_index(drop=True)], axis=1)
    for _col in [
        "fluency",
        "grammar",
        "tone",
        "length",
        "grounding",
        "latency",
        "cost",
        "final_score",
    ]:
        df[_col] = ""

    df.to_excel(ASSIGNMENT_XLSX_PATH, index=False)
    print(
        f"Done. Mean latency: {results_df['latency_ms'].mean():.0f} ms | "
        f"Total cost: ${results_df['cost_usd'].sum():.6f}"
    )
    print(f"Saved {len(df)} rows to {ASSIGNMENT_XLSX_PATH}")

Source: loaded existing artifact /Users/zakhar/Documents/code/learning-ai-nebius-ai-engineering-2026/w2-ai-product/outputs/assignment_01.xlsx


## Sample Outputs

In [ ]:
mo.ui.table(
    df[["product_name", "generated_description", "latency_ms", "input_tokens", "output_tokens", "cost_usd"]].head(10),
    label="First 10 generated descriptions"
)

,_marimo_row_id,product_name,generated_description,latency_ms,input_tokens,output_tokens,cost_usd
0,0,Apple iPhone 15 Pro,Experience the future of smartphone technology...,8572.0,214,112,0.000011
1,1,Samsung Galaxy S24 Ultra,Elevate your mobile experience with the Samsun...,5682.3,217,112,0.000011
2,2,Google Pixel 8 Pro,Capture life's moments with the Google Pixel 8...,4949.9,215,101,0.000010
3,3,Sony WH‑1000XM5 Headphones,Immerse yourself in pure sound with the Sony W...,4303.5,214,112,0.000011
4,4,Bose QuietComfort Ultra Earbuds,Immerse yourself in pure sound with the Bose Q...,2726.2,208,96,0.000010
5,5,Amazon Echo Dot (5th Gen),"Meet the Amazon Echo Dot (5th Gen), your smart...",2955.7,204,107,0.000010
6,6,Dell XPS 13 9310 Laptop,Experience the ultimate in portability and per...,4076.0,222,113,0.000011
7,7,Apple MacBook Air 13″ (M3),Get ready to experience the future of computin...,3732.5,213,109,0.000011
8,8,Microsoft Surface Pro 10,Unlock your full potential with the Microsoft ...,3219.4,206,95,0.000010
9,9,Garmin Forerunner 255,Take your training to the next level with the ...,3724.9,216,106,0.000011
